In [17]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
import os

In [3]:
load_dotenv()

True

# Step 1a. INDEXING ( Document ingestion )

In [4]:
# "YtHdaXuOAks"
video_id = "iUh-8yjycHU" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

[Music] Everyone's like, "What? Fall through the basket?" Competition. Let's talk about that. That's all connected. We're not doing a lower speeds today. No, no, no. So, as you can see, 250 again. Yay. What's up, ladies and gentlemen, and welcome to this POV review by Outtopell. My name is Martin and today we're looking at an Audi that solved an autobomb problem, an outtop problem. Like 10 years ago or 5 years ago even, all the mid-level BMWs, Mercedes, Audi's could do 250 km an hour. Then they introduced all the plug-in hybrids and we could only do like 220 or 230. Now, good news for the autobon lovers. The A5, the new one, the plug-in hybrids can do 250 again. Yay. So, great job Audi. So, this is actually the base level version 40 plug-in hybrid. So, this is 299 horsepower and even this will do 250. There's also a 367 horsepower version, which obviously also does 250. And of course, there's also the Audi S5, but that's a V6 mild hybrid. This is a real true plug-in hybrid. It can do 1

In [5]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Music]', start=0.0, duration=3.28), FetchedTranscriptSnippet(text='Everyone\'s like, "What?', start=0.32, duration=5.12), FetchedTranscriptSnippet(text='Fall through the basket?" Competition.', start=3.28, duration=3.68), FetchedTranscriptSnippet(text="Let's talk about that. That's all", start=5.44, duration=3.36), FetchedTranscriptSnippet(text="connected. We're not doing a lower", start=6.96, duration=4.559), FetchedTranscriptSnippet(text='speeds today. No, no, no. So, as you can', start=8.8, duration=8.479), FetchedTranscriptSnippet(text='see, 250 again. Yay.', start=11.519, duration=7.281), FetchedTranscriptSnippet(text="What's up, ladies and gentlemen, and", start=17.279, duration=4.241), FetchedTranscriptSnippet(text='welcome to this POV review by Outtopell.', start=18.8, duration=4.319), FetchedTranscriptSnippet(text="My name is Martin and today we're", start=21.52, duration=3.999), FetchedTranscriptSnippet(text='looking

# Step 1b. INDEXING ( Text splitting )

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [7]:
len(chunks)

12

In [8]:
chunks[10]

Document(metadata={}, page_content='corner and then these bumps come into play and that\'s where a lot of suspension setups fall through the basket. I love doing that. Everyone\'s like, "What? Fall through the basket?" Go look it up. [Music] But this suspension really good. Audi has made a very good car with this new A5. Just get it as an Avanton. I think it\'s much prettier as an Avanton and much more practical, of course. But this drivetrain fabulous. Really, I would personally get the uh extra 68 horsepower because that was just fast, quick, it felt like a bit too much, which is always good for a daily driver. So, yeah, all in all, great car. Great car. If you live in Germany, you want to go a bit faster when it quiets down on the autobon, this is it. It does truly make a difference if you can do 230 or 250. It does feel I don\'t know why, but it does make a difference. Okay, guys. Thank you for watching. Hope you like the video. Hope you like the car. And if you don\'t, go watch an

# Step 1c & 1d. INDEXING ( Embedding generation and storing in vector store )

In [11]:
os.environ["GOOGLE_API_KEY"] = ""
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
)
vector_store = FAISS.from_documents(chunks, embeddings)

In [12]:
vector_store.index_to_docstore_id

{0: '40dc6e5f-12c9-46be-bf02-bfcdbc5fe85e',
 1: '3fd5c8c3-fa40-4c13-a306-108ee24aee58',
 2: '5f2d2b93-a202-48ec-a485-6de458e3d3a3',
 3: '552e0ee3-324f-4c38-babf-68e910f74067',
 4: '1cd95f6a-6039-4875-9fc9-ba6462c217a5',
 5: 'd81109af-3ea8-4ac1-b028-25f045e5a67a',
 6: '1f49a2df-7628-4711-9ea3-f24ac6d6643f',
 7: '56ddd9b2-6068-4002-8d3e-02bf90dfde9d',
 8: 'b8075b18-6a71-4658-bce8-d23f6c4484d2',
 9: 'e1864b4e-da6c-4c79-a4b0-4b875db991b8',
 10: '97b2a4bb-6820-4831-a911-5473654cd922',
 11: '814fa8bf-3cc7-4d1b-ac84-10c82f7eac57'}

# STEP 2 - RETRIEVAL

In [13]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20})

In [14]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F94C5EB2D0>, search_type='mmr', search_kwargs={'k': 4, 'fetch_k': 20})

In [15]:
retriever.invoke('what is audi a5')

[Document(id='97b2a4bb-6820-4831-a911-5473654cd922', metadata={}, page_content='corner and then these bumps come into play and that\'s where a lot of suspension setups fall through the basket. I love doing that. Everyone\'s like, "What? Fall through the basket?" Go look it up. [Music] But this suspension really good. Audi has made a very good car with this new A5. Just get it as an Avanton. I think it\'s much prettier as an Avanton and much more practical, of course. But this drivetrain fabulous. Really, I would personally get the uh extra 68 horsepower because that was just fast, quick, it felt like a bit too much, which is always good for a daily driver. So, yeah, all in all, great car. Great car. If you live in Germany, you want to go a bit faster when it quiets down on the autobon, this is it. It does truly make a difference if you can do 230 or 250. It does feel I don\'t know why, but it does make a difference. Okay, guys. Thank you for watching. Hope you like the video. Hope you 

# STEP 3 - AUGMENTATION

In [19]:
model = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash')

prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

question          = "Was there any discussion about audi a5 ? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [20]:
retrieved_docs

[Document(id='97b2a4bb-6820-4831-a911-5473654cd922', metadata={}, page_content='corner and then these bumps come into play and that\'s where a lot of suspension setups fall through the basket. I love doing that. Everyone\'s like, "What? Fall through the basket?" Go look it up. [Music] But this suspension really good. Audi has made a very good car with this new A5. Just get it as an Avanton. I think it\'s much prettier as an Avanton and much more practical, of course. But this drivetrain fabulous. Really, I would personally get the uh extra 68 horsepower because that was just fast, quick, it felt like a bit too much, which is always good for a daily driver. So, yeah, all in all, great car. Great car. If you live in Germany, you want to go a bit faster when it quiets down on the autobon, this is it. It does truly make a difference if you can do 230 or 250. It does feel I don\'t know why, but it does make a difference. Okay, guys. Thank you for watching. Hope you like the video. Hope you 

In [21]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'corner and then these bumps come into play and that\'s where a lot of suspension setups fall through the basket. I love doing that. Everyone\'s like, "What? Fall through the basket?" Go look it up. [Music] But this suspension really good. Audi has made a very good car with this new A5. Just get it as an Avanton. I think it\'s much prettier as an Avanton and much more practical, of course. But this drivetrain fabulous. Really, I would personally get the uh extra 68 horsepower because that was just fast, quick, it felt like a bit too much, which is always good for a daily driver. So, yeah, all in all, great car. Great car. If you live in Germany, you want to go a bit faster when it quiets down on the autobon, this is it. It does truly make a difference if you can do 230 or 250. It does feel I don\'t know why, but it does make a difference. Okay, guys. Thank you for watching. Hope you like the video. Hope you like the car. And if you don\'t, go watch another one. Subscribe to the channel

In [22]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text='\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don\'t know.\n\n      corner and then these bumps come into play and that\'s where a lot of suspension setups fall through the basket. I love doing that. Everyone\'s like, "What? Fall through the basket?" Go look it up. [Music] But this suspension really good. Audi has made a very good car with this new A5. Just get it as an Avanton. I think it\'s much prettier as an Avanton and much more practical, of course. But this drivetrain fabulous. Really, I would personally get the uh extra 68 horsepower because that was just fast, quick, it felt like a bit too much, which is always good for a daily driver. So, yeah, all in all, great car. Great car. If you live in Germany, you want to go a bit faster when it quiets down on the autobon, this is it. It does truly make a difference if you can do 230 or 250. It does feel I don\'t

# STEP 4 - GENERATION

In [23]:
answer = model.invoke(final_prompt)
print(answer.content)

Yes, the Audi A5 was discussed.

Here's what was mentioned:
*   Audi has made a very good car with the new A5.
*   It's suggested to get it as an Avanton, as it's considered prettier and more practical.
*   The new A5 plug-in hybrids can reach 250 km/h, which is an improvement over previous plug-in hybrids that could only do 220 or 230 km/h.
*   The base level version 40 plug-in hybrid has 299 horsepower and can do 250 km/h.
*   There is also a 367 horsepower version that also reaches 250 km/h.
*   The Audi S5 is mentioned as a V6 mild hybrid, differentiating it from the A5 plug-in hybrids.
*   There's a 10% discount on RaceBox products using the code "Audi A5".


# STEP 5 - Building a CHAIN

In [24]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [25]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [26]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [27]:
parallel_chain.invoke('Was there any discussion about audi a5 ? if yes then what was discussed')

{'context': 'corner and then these bumps come into play and that\'s where a lot of suspension setups fall through the basket. I love doing that. Everyone\'s like, "What? Fall through the basket?" Go look it up. [Music] But this suspension really good. Audi has made a very good car with this new A5. Just get it as an Avanton. I think it\'s much prettier as an Avanton and much more practical, of course. But this drivetrain fabulous. Really, I would personally get the uh extra 68 horsepower because that was just fast, quick, it felt like a bit too much, which is always good for a daily driver. So, yeah, all in all, great car. Great car. If you live in Germany, you want to go a bit faster when it quiets down on the autobon, this is it. It does truly make a difference if you can do 230 or 250. It does feel I don\'t know why, but it does make a difference. Okay, guys. Thank you for watching. Hope you like the video. Hope you like the car. And if you don\'t, go watch another one. Subscribe to

In [28]:
parser = StrOutputParser()

In [29]:
main_chain = parallel_chain | prompt | model | parser

In [30]:
main_chain.invoke('Can you summarize the video')

'This video is a POV review of the new Audi A5 plug-in hybrid by Martin from Outtopell.\n\nThe main highlight is that the new A5 plug-in hybrids can once again reach 250 km/h, solving a previous "autobahn problem" where newer plug-in hybrids were limited to lower speeds. The base level 40 plug-in hybrid, with 299 horsepower, can achieve this speed, as can a 367 horsepower version. This model is noted as a "real true plug-in hybrid," distinguishing it from the V6 mild hybrid Audi S5.\n\nIn terms of performance, the car accelerates from 0-100 km/h in 5.84-5.85 seconds, which is slightly faster than the claimed 5.9 seconds and is very consistent. The gearbox is a dual-clutch unit, and while it feels "alive" in general, manual shifting is described as very laggy, especially downshifts. The reviewer recommends using "Sport" mode, which keeps both the electric and petrol engines engaged, providing a constant 300 horsepower. In "D" mode, the combustion engine recedes into the background, rely